# InfoGAN + OpenMax — Kaggle Training

Adapted for **Kaggle Free Tier** (T4 x2 GPU, 13GB RAM, 12h session limit).

**Key optimizations:**
1. Streaming `tf.data` pipeline — no full dataset in RAM
2. Mixed precision (float16 compute, float32 outputs)
3. Checkpointing every 5 epochs with auto-resume
4. Gradient clipping + label smoothing for stability
5. Reduced Python overhead in training loop
6. Optional early stopping
7. Periodic sample generation + history save
8. `he_normal` init, scaler range [-1,1] for tanh output

**Setup:** This notebook expects the helper `.py` files uploaded as a Kaggle dataset.
See the setup instructions in the first code cell.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 0. KAGGLE SETUP
# ═══════════════════════════════════════════════════════════════════════════
import sys, os

# ── Add helper scripts dataset to path ──
# Upload infogan_model_kaggle.py, nids_utils_kaggle.py, openmax_kaggle.py
# as a Kaggle dataset (e.g. "nids-helper-scripts") and reference it here:
HELPERS_DIR = "/kaggle/input/nids-helper-scripts"
sys.path.insert(0, HELPERS_DIR)

import infogan_model_kaggle as igo
import nids_utils_kaggle as nu
import openmax_kaggle as om

# Enable mixed precision + memory growth BEFORE building any model
igo.setup_kaggle()

import numpy as np
import pandas as pd
import json
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tensorflow as tf
import keras

## 1. Configuration

All tuneable parameters in one place.

In [ ]:
# ── Paths ──
ALL_CSV_PATHS = [nu.MONDAY_CSV] + nu.ATTACK_FILES
SAVE_DIR = os.path.join(nu.MODELS_ROOT, "infogan", "kaggle_run")
CHECKPOINT_DIR = os.path.join(SAVE_DIR, "checkpoints")
os.makedirs(SAVE_DIR, exist_ok=True)

# ── Open-set split ──
KNOWN_CLASSES = [0, 1, 2, 3, 5, 6]  # Hold out class 4 (Infiltration) as unknown
UNKNOWN_CLASSES = [4]
NUM_KNOWN = len(KNOWN_CLASSES)

# ── Training hyperparams ──
NOISE_DIM = 100
LR = 0.0002
LAMBDA_INFO = 1.0
BATCH_SIZE = 512       # larger batch = fewer steps = faster epochs (model is tiny)
EPOCHS = 200
SHUFFLE_BUFFER = 50000

# ── OpenMax params ──
TAIL_SIZE = 20
ALPHA_RANK = 3
DISTANCE_TYPE = "cosine"

# ── Training options ──
PRINT_EVERY = 10
SAMPLE_EVERY = 10
CKPT_EVERY = 5
EARLY_STOP = False  # Set True if you want auto-stop on G-loss plateau

known_class_names = [igo.CLASS_NAMES[c] for c in KNOWN_CLASSES]
print(f"Known classes ({NUM_KNOWN}): {known_class_names}")
print(f"Unknown classes: {[igo.CLASS_NAMES[c] for c in UNKNOWN_CLASSES]}")
print(f"Save dir: {SAVE_DIR}")

## 2. Fit Scaler (streaming, feature_range=[-1,1])

Two-pass approach: scan all CSVs for global min/max without loading everything into RAM.
Range [-1,1] matches the Generator's tanh output.

In [ ]:
SCALER_PATH = os.path.join(nu.PREPROCESSING_DIR, "scaler_infogan_kaggle.pkl")
FEATURE_COLS_PATH = os.path.join(nu.PREPROCESSING_DIR, "infogan_kaggle_feature_cols.json")

# Check if scaler already exists (resume-friendly)
if os.path.exists(SCALER_PATH) and os.path.exists(FEATURE_COLS_PATH):
    print("Loading existing scaler...")
    scaler = joblib.load(SCALER_PATH)
    with open(FEATURE_COLS_PATH) as f:
        feature_cols = json.load(f)
    print(f"Scaler loaded: {len(feature_cols)} features, range={scaler.feature_range}")
else:
    print("Fitting scaler (streaming, range=[-1,1])...")
    scaler, feature_cols = igo.fit_scaler_streaming(ALL_CSV_PATHS, feature_range=(-1, 1))
    os.makedirs(os.path.dirname(SCALER_PATH), exist_ok=True)
    joblib.dump(scaler, SCALER_PATH)
    with open(FEATURE_COLS_PATH, "w") as f:
        json.dump(feature_cols, f)
    print(f"Scaler saved: {len(feature_cols)} features")

## 3. Pre-process CSVs to Numpy (one-time)

Convert all CSVs to numpy arrays **once**, then train from those. This eliminates the CPU bottleneck of re-reading and parsing CSVs every epoch.

In [ ]:
# Clean old cached data if re-running with updated code
import glob as _glob
old_npy = _glob.glob(os.path.join(nu.PREPROCESSING_DIR, "*.npy"))
if old_npy:
    print(f"Cleaning {len(old_npy)} cached .npy files from previous run...")
    for f in old_npy:
        os.remove(f)

# Pre-process CSVs -> numpy (runs once, cached on disk)
images_path, labels_path, total_known_samples = igo.preprocess_csvs_to_numpy(
    ALL_CSV_PATHS, scaler, feature_cols,
    known_classes=KNOWN_CLASSES,
    save_dir=nu.PREPROCESSING_DIR,
)

# Build fast tf.data dataset from numpy (loaded into RAM, cached by TF)
train_dataset_with_labels, _ = igo.build_dataset_from_numpy(
    images_path, labels_path, BATCH_SIZE,
)

# InfoGAN training is unsupervised — strip labels
train_dataset = train_dataset_with_labels.map(lambda img, lab: img, num_parallel_calls=tf.data.AUTOTUNE)

steps_per_epoch = total_known_samples // BATCH_SIZE
print(f"Total known samples: {total_known_samples:,}")
print(f"Batch size:          {BATCH_SIZE}")
print(f"Steps per epoch:     {steps_per_epoch:,}")

## 4. Build Models + Trainer

In [ ]:
backbone = igo.build_shared_backbone()
discriminator = igo.build_discriminator(backbone)
classifier = igo.build_classifier(backbone, num_classes=NUM_KNOWN)
generator = igo.build_generator(NOISE_DIM, num_classes=NUM_KNOWN)

print(f"Backbone params:      {backbone.count_params():,}")
print(f"Discriminator params: {discriminator.count_params():,}")
print(f"Classifier params:    {classifier.count_params():,}")
print(f"Generator params:     {generator.count_params():,}")

In [ ]:
trainer = igo.InfoGANTrainer(
    generator=generator,
    discriminator=discriminator,
    classifier=classifier,
    backbone=backbone,
    noise_dim=NOISE_DIM,
    num_classes=NUM_KNOWN,
    lr=LR,
    lambda_info=LAMBDA_INFO,
    checkpoint_dir=CHECKPOINT_DIR,
)

## 5. Train InfoGAN

- Checkpoints every 5 epochs (auto-resume on session restart)
- Sample generation every 10 epochs
- History saved periodically
- Mixed precision active (speedup on T4, moderate benefit on P100)

In [ ]:
history = trainer.fit(
    dataset=train_dataset,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    save_dir=SAVE_DIR,
    print_every=PRINT_EVERY,
    sample_every=SAMPLE_EVERY,
    ckpt_every=CKPT_EVERY,
    early_stop=EARLY_STOP,
)

In [ ]:
# Plot training losses
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history["d_loss"]); axes[0].set_title("Discriminator Loss"); axes[0].set_xlabel("Epoch")
axes[1].plot(history["g_loss"]); axes[1].set_title("Generator Loss"); axes[1].set_xlabel("Epoch")
axes[2].plot(history["q_loss"]); axes[2].set_title("Q Loss (Mutual Info)"); axes[2].set_xlabel("Epoch")
fig.suptitle("InfoGAN Training Losses (Kaggle)")
fig.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, "training_losses.png"), dpi=150)
plt.show()

# Timing summary
total_time = sum(history["epoch_time"])
print(f"\nTotal training time: {total_time/60:.1f} min ({total_time/3600:.2f} hours)")
print(f"Average epoch time: {np.mean(history['epoch_time']):.1f}s")

## 6. Load Evaluation Data

For OpenMax fitting and metrics, we need numpy arrays. Load in chunks to control memory.

In [ ]:
from sklearn.model_selection import train_test_split

# Reuse the pre-processed numpy arrays (already on disk from step 3)
print("Loading pre-processed known class data...")
X_known = np.load(images_path)
y_known = np.load(labels_path)
print(f"Known data: {X_known.shape[0]:,} samples")

# Train/test split
X_train_eval, X_test_known, y_train_eval, y_test_known = train_test_split(
    X_known, y_known, test_size=0.2, random_state=42, stratify=y_known
)
del X_known, y_known  # free

# Load unknown class data (separate preprocessing)
if UNKNOWN_CLASSES:
    unk_img_path = os.path.join(nu.PREPROCESSING_DIR, "unknown_images.npy")
    unk_lab_path = os.path.join(nu.PREPROCESSING_DIR, "unknown_labels.npy")
    if os.path.exists(unk_img_path):
        print("Loading cached unknown class data...")
        X_test_unknown = np.load(unk_img_path)
    else:
        print("Loading unknown class data from CSVs...")
        X_test_unknown, _ = igo.build_image_label_arrays_streamed(
            ALL_CSV_PATHS, scaler, feature_cols, known_classes=UNKNOWN_CLASSES
        )
        np.save(unk_img_path, X_test_unknown)
    print(f"Unknown data: {X_test_unknown.shape[0]:,} samples")
else:
    X_test_unknown = np.empty((0, 11, 11, 1), dtype=np.float32)

print(f"\nTrain (for OpenMax): {X_train_eval.shape[0]:,}")
print(f"Test (known):        {X_test_known.shape[0]:,}")
print(f"Test (unknown):      {X_test_unknown.shape[0]:,}")

## 7. Classifier Q — Pseudo-label Evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from scipy.optimize import linear_sum_assignment

# Predict with classifier Q
y_pred_train = np.argmax(classifier.predict(X_train_eval, batch_size=1024, verbose=0), axis=1)
y_pred_test = np.argmax(classifier.predict(X_test_known, batch_size=1024, verbose=0), axis=1)

# Hungarian matching (InfoGAN is unsupervised — cluster IDs don't match class IDs)
def find_best_label_mapping(y_true, y_pred, num_classes):
    cost_matrix = np.zeros((num_classes, num_classes))
    for true_c in range(num_classes):
        for pred_c in range(num_classes):
            mask = y_true == true_c
            cost_matrix[true_c, pred_c] = -np.sum(y_pred[mask] == pred_c)
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    return {col_ind[i]: row_ind[i] for i in range(num_classes)}

mapping = find_best_label_mapping(y_train_eval, y_pred_train, NUM_KNOWN)
print("Label mapping (cluster -> true):")
for pred_c, true_c in sorted(mapping.items()):
    print(f"  Cluster {pred_c} -> {known_class_names[true_c]}")

y_pred_test_mapped = np.array([mapping[p] for p in y_pred_test])
y_pred_train_mapped = np.array([mapping[p] for p in y_pred_train])

print(f"\nClassifier Q Test Accuracy: {accuracy_score(y_test_known, y_pred_test_mapped):.4f}")
print(classification_report(y_test_known, y_pred_test_mapped, target_names=known_class_names))

## 8. Fit OpenMax

In [ ]:
# Get activation vectors (128-dim penultimate layer)
avs_train = trainer.get_activation_vectors(X_train_eval, batch_size=1024)
print(f"Activation vectors: {avs_train.shape}")

# Compute MAVs using mapped labels
mavs, class_avs = om.compute_mavs(avs_train, y_pred_train_mapped, NUM_KNOWN)
print(f"MAVs computed for {len(mavs)} classes")

# Fit Weibull
distances = om.compute_distances(class_avs, mavs, distance_type=DISTANCE_TYPE)
weibull_params = om.fit_weibull(distances, tail_size=TAIL_SIZE)

for cls, params in weibull_params.items():
    if params:
        print(f"  {known_class_names[cls]}: shape={params[0]:.4f}, scale={params[2]:.6f}")
    else:
        print(f"  {known_class_names[cls]}: FAILED")

## 9. Open-Set Model Evaluation (Vectorized OpenMax)

In [ ]:
import time

# Known test data
avs_test_known = trainer.get_activation_vectors(X_test_known, batch_size=1024)

t0 = time.time()
preds_known, probs_known = om.openmax_predict(
    avs_test_known, mavs, weibull_params,
    alpha_rank=ALPHA_RANK, num_classes=NUM_KNOWN,
    distance_type=DISTANCE_TYPE,
)
print(f"OpenMax on {len(avs_test_known):,} known samples: {time.time()-t0:.1f}s")

known_correct = np.sum((preds_known != NUM_KNOWN) & (preds_known == y_test_known))
known_as_unknown = np.sum(preds_known == NUM_KNOWN)
print(f"Correctly classified: {known_correct:,} / {len(y_test_known):,}")
print(f"False unknowns:      {known_as_unknown:,}")

In [ ]:
# Unknown test data
if len(X_test_unknown) > 0:
    avs_test_unknown = trainer.get_activation_vectors(X_test_unknown, batch_size=1024)
    preds_unknown, probs_unknown = om.openmax_predict(
        avs_test_unknown, mavs, weibull_params,
        alpha_rank=ALPHA_RANK, num_classes=NUM_KNOWN,
        distance_type=DISTANCE_TYPE,
    )
    unknown_detected = np.sum(preds_unknown == NUM_KNOWN)
    print(f"Unknown detection: {unknown_detected}/{len(preds_unknown)} ({unknown_detected/len(preds_unknown):.4f})")
else:
    print("No unknown test data.")

## 10. Comprehensive Metrics

In [ ]:
from sklearn.metrics import (
    precision_recall_fscore_support,
    roc_curve, auc,
    multilabel_confusion_matrix,
    precision_recall_curve, average_precision_score,
)
from sklearn.preprocessing import label_binarize

# Combine known + unknown
all_class_names = known_class_names + (["Unknown"] if len(X_test_unknown) > 0 else [])
num_all = NUM_KNOWN + (1 if len(X_test_unknown) > 0 else 0)

if len(X_test_unknown) > 0:
    y_true_all = np.concatenate([y_test_known, np.full(len(X_test_unknown), NUM_KNOWN)])
    y_pred_all = np.concatenate([preds_known, preds_unknown])
    probs_all = np.concatenate([probs_known, probs_unknown])
else:
    y_true_all = y_test_known
    y_pred_all = preds_known
    probs_all = probs_known

# ── Per-class Precision, Recall, F1, FPR ──
prec, rec, f1, support = precision_recall_fscore_support(
    y_true_all, y_pred_all, labels=list(range(num_all)), zero_division=0
)

mcm = multilabel_confusion_matrix(y_true_all, y_pred_all, labels=list(range(num_all)))
fpr_per_class = []
for i in range(num_all):
    tn, fp, fn, tp = mcm[i].ravel()
    fpr_per_class.append(fp / (fp + tn) if (fp + tn) > 0 else 0.0)

metrics_df = pd.DataFrame({
    "Class": all_class_names,
    "Precision": prec, "Recall": rec, "F1-Score": f1,
    "FPR": fpr_per_class, "Support": support.astype(int),
})
metrics_df = pd.concat([metrics_df, pd.DataFrame([{
    "Class": "Macro Avg", "Precision": prec.mean(), "Recall": rec.mean(),
    "F1-Score": f1.mean(), "FPR": np.mean(fpr_per_class), "Support": support.sum(),
}, {
    "Class": "Weighted Avg",
    "Precision": np.average(prec, weights=support),
    "Recall": np.average(rec, weights=support),
    "F1-Score": np.average(f1, weights=support),
    "FPR": np.average(fpr_per_class, weights=support),
    "Support": support.sum(),
}])], ignore_index=True)

print("=" * 80)
print("  O-S MODEL — COMPLETE PERFORMANCE METRICS")
print("=" * 80)
print(metrics_df.to_string(index=False, float_format="%.4f"))
print(f"\nOverall Accuracy: {accuracy_score(y_true_all, y_pred_all):.4f}")

In [ ]:
# ── ROC Curves ──
y_true_bin = label_binarize(y_true_all, classes=list(range(num_all)))
if y_true_bin.shape[1] == 1:
    y_true_bin = np.hstack([1 - y_true_bin, y_true_bin])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = plt.cm.tab10(np.linspace(0, 1, num_all))
auc_scores = {}

ax = axes[0]
for i in range(num_all):
    if y_true_bin[:, i].sum() == 0:
        continue
    fpr_i, tpr_i, _ = roc_curve(y_true_bin[:, i], probs_all[:, i])
    auc_i = auc(fpr_i, tpr_i)
    auc_scores[all_class_names[i]] = auc_i
    ax.plot(fpr_i, tpr_i, color=colors[i], lw=2, label=f"{all_class_names[i]} (AUC={auc_i:.4f})")
ax.plot([0,1],[0,1], "k--", lw=1, alpha=0.5)
ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.set_title("ROC — One-vs-Rest")
ax.legend(loc="lower right", fontsize=8); ax.grid(True, alpha=0.3)

# Macro-average ROC
ax = axes[1]
all_fpr = np.linspace(0, 1, 200)
mean_tpr = np.zeros_like(all_fpr)
valid = 0
for i in range(num_all):
    if y_true_bin[:, i].sum() == 0: continue
    fpr_i, tpr_i, _ = roc_curve(y_true_bin[:, i], probs_all[:, i])
    mean_tpr += np.interp(all_fpr, fpr_i, tpr_i); valid += 1
mean_tpr /= max(valid, 1)
macro_auc = auc(all_fpr, mean_tpr)
ax.plot(all_fpr, mean_tpr, "navy", lw=2, label=f"Macro-Avg (AUC={macro_auc:.4f})")
ax.fill_between(all_fpr, 0, mean_tpr, alpha=0.1, color="navy")
ax.plot([0,1],[0,1], "k--", lw=1, alpha=0.5)
ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.set_title("Macro-Average ROC")
ax.legend(); ax.grid(True, alpha=0.3)

fig.suptitle("O-S Model — ROC/AUC", fontsize=14)
fig.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, "roc_curves.png"), dpi=150)
plt.show()

print("\nAUC per class:")
for c, a in auc_scores.items(): print(f"  {c}: {a:.4f}")
print(f"  Macro-Avg: {macro_auc:.4f}")

In [ ]:
# ── Precision-Recall Curves ──
fig, ax = plt.subplots(figsize=(10, 6))
ap_scores = {}
for i in range(num_all):
    if y_true_bin[:, i].sum() == 0: continue
    prec_i, rec_i, _ = precision_recall_curve(y_true_bin[:, i], probs_all[:, i])
    ap_i = average_precision_score(y_true_bin[:, i], probs_all[:, i])
    ap_scores[all_class_names[i]] = ap_i
    ax.plot(rec_i, prec_i, color=colors[i], lw=2, label=f"{all_class_names[i]} (AP={ap_i:.4f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves"); ax.legend(loc="lower left", fontsize=8)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, "pr_curves.png"), dpi=150)
plt.show()

print("\nAverage Precision:")
for c, a in ap_scores.items(): print(f"  {c}: {a:.4f}")
print(f"  mAP: {np.mean(list(ap_scores.values())):.4f}")

In [ ]:
# ── Open-Set Specific Metrics ──
print("=" * 60)
print("  OPEN-SET RECOGNITION METRICS")
print("=" * 60)

known_acc = np.mean((preds_known != NUM_KNOWN) & (preds_known == y_test_known))
known_rejection = np.mean(preds_known == NUM_KNOWN)
print(f"Known Classification Accuracy: {known_acc:.4f}")
print(f"Known Rejection Rate:          {known_rejection:.4f}")

if len(X_test_unknown) > 0:
    udr = np.mean(preds_unknown == NUM_KNOWN)
    print(f"Unknown Detection Rate:        {udr:.4f}")
    print(f"False Known Rate:              {1 - udr:.4f}")
    h_score = 2 * known_acc * udr / (known_acc + udr) if (known_acc + udr) > 0 else 0
    print(f"H-Score (KCA & UDR harmonic):  {h_score:.4f}")

# ── Confusion Matrix ──
cm = confusion_matrix(y_true_all, y_pred_all)
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
ax.set_title("O-S Model Confusion Matrix")
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ticks = np.arange(num_all)
ax.set_xticks(ticks); ax.set_xticklabels(all_class_names, rotation=45, ha="right")
ax.set_yticks(ticks); ax.set_yticklabels(all_class_names)
fig.colorbar(im); fig.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, "confusion_matrix.png"), dpi=150)
plt.show()

## 11. Save All Artifacts

In [ ]:
# Save Keras models
generator.save(os.path.join(SAVE_DIR, "generator.keras"))
discriminator.save(os.path.join(SAVE_DIR, "discriminator.keras"))
classifier.save(os.path.join(SAVE_DIR, "classifier.keras"))
backbone.save(os.path.join(SAVE_DIR, "backbone.keras"))

# Save OpenMax params
openmax_data = {
    "mavs": {str(k): v.tolist() for k, v in mavs.items()},
    "weibull_params": {str(k): list(v) if v else None for k, v in weibull_params.items()},
    "label_mapping": {str(k): int(v) for k, v in mapping.items()},
    "known_classes": KNOWN_CLASSES, "unknown_classes": UNKNOWN_CLASSES,
    "tail_size": TAIL_SIZE, "alpha_rank": ALPHA_RANK,
    "distance_type": DISTANCE_TYPE, "num_known": NUM_KNOWN,
}
with open(os.path.join(SAVE_DIR, "openmax_params.json"), "w") as f:
    json.dump(openmax_data, f, indent=2)

# Save MAVs
np.savez(os.path.join(SAVE_DIR, "openmax_mavs.npz"),
         **{f"class_{k}": v for k, v in mavs.items()})

# Save metrics
metrics_df.to_csv(os.path.join(SAVE_DIR, "performance_metrics.csv"), index=False)
pd.DataFrame({"Class": list(auc_scores.keys()), "AUC": list(auc_scores.values()),
              "AP": [ap_scores.get(c, np.nan) for c in auc_scores.keys()]
}).to_csv(os.path.join(SAVE_DIR, "auc_ap_scores.csv"), index=False)

# Save metadata
meta = {
    "model_type": "infogan_openmax_os_kaggle",
    "platform": "kaggle",
    "optimizations": ["mixed_precision", "gradient_clipping", "label_smoothing",
                       "he_normal_init", "scaler_range_neg1_1", "checkpointing",
                       "streaming_tf_data"],
    "noise_dim": NOISE_DIM, "num_known": NUM_KNOWN,
    "known_classes": KNOWN_CLASSES, "unknown_classes": UNKNOWN_CLASSES,
    "known_class_names": known_class_names,
    "epochs": EPOCHS, "batch_size": BATCH_SIZE, "lr": LR,
    "lambda_info": LAMBDA_INFO, "tail_size": TAIL_SIZE,
    "alpha_rank": ALPHA_RANK, "distance_type": DISTANCE_TYPE,
    "num_features": len(feature_cols),
    "total_training_time_min": sum(history["epoch_time"]) / 60,
    "final_d_loss": history["d_loss"][-1],
    "final_g_loss": history["g_loss"][-1],
    "final_q_loss": history["q_loss"][-1],
    "overall_accuracy": float(accuracy_score(y_true_all, y_pred_all)),
    "macro_auc": float(macro_auc),
}
with open(os.path.join(SAVE_DIR, "meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

print(f"\nAll artifacts saved -> {SAVE_DIR}")
for fname in sorted(os.listdir(SAVE_DIR)):
    if not os.path.isdir(os.path.join(SAVE_DIR, fname)):
        print(f"  {fname}")

## 12. Hyperparameter Sweep (Optional)

In [ ]:
if len(X_test_unknown) > 0:
    results = []
    for ts in [5, 10, 20, 50]:
        wp = om.fit_weibull(distances, tail_size=ts)
        for ar in [1, 2, 3, 5]:
            pk, _ = om.openmax_predict(avs_test_known, mavs, wp, ar, NUM_KNOWN, DISTANCE_TYPE)
            pu, _ = om.openmax_predict(avs_test_unknown, mavs, wp, ar, NUM_KNOWN, DISTANCE_TYPE)
            results.append({
                "tail_size": ts, "alpha_rank": ar,
                "known_acc": np.mean((pk != NUM_KNOWN) & (pk == y_test_known)),
                "unknown_det": np.mean(pu == NUM_KNOWN),
            })
    sweep_df = pd.DataFrame(results)
    sweep_df.to_csv(os.path.join(SAVE_DIR, "hyperparameter_sweep.csv"), index=False)
    print(sweep_df.to_string(index=False))
else:
    print("No unknown data — skipping sweep.")

## Summary

| Optimization | Effect |
|---|---|
| Mixed precision (float16) | ~1.5-2x faster training, less GPU memory |
| Streaming tf.data | ~1.3GB RAM saved (no full image array) |
| Checkpointing every 5 epochs | Survives Kaggle session restarts |
| Gradient clipping (clipnorm=1.0) | Prevents gradient explosion |
| Label smoothing (0.9) | Prevents discriminator overconfidence |
| `he_normal` init | Better convergence for ReLU/LeakyReLU |
| Scaler range [-1,1] | Matches tanh output, no wasted dynamic range |
| Vectorized OpenMax | ~100x faster than per-sample Python loop |
| Periodic sample gen | Visual monitoring of Generator quality |

**Estimated time on T4 x2:** ~1.5-2.5 hours for 200 epochs

**Download artifacts:** All outputs are saved to `/kaggle/working/` and will be available in the Output tab after the notebook finishes.